In [2]:
import pandas as pd
import numpy as np
import re

def normalize_price(value):
    """
    Chuẩn hóa giá:
    - '19.990.000₫'  -> '19990000'
    - '12,9 triệu'   -> '12900000'
    - 'Liên hệ'      -> NaN
    """
    if pd.isna(value):
        return np.nan

    s = str(value).lower()

    if any(x in s for x in ["liên hệ", "đang cập nhật", "call", "contact"]):
        return np.nan

    match = re.search(r"([\d.,]+)\s*triệu", s)
    if match:
        return str(int(float(match.group(1).replace(",", ".")) * 1_000_000))

    digits = re.sub(r"[^\d]", "", s)
    return digits if digits else np.nan

def preprocess_product_file(
    input_path,
    output_path,
    keep_cols,
    category_value,
    desc_col_name="description"
):
    df = pd.read_excel(input_path)
    print(f"📂 Đang mở file: {input_path}")

    # 1️⃣ Category (thêm hoặc ghi đè)
    df["category"] = category_value

    # 2️⃣ Chuẩn hóa price (KHÔNG tạo price_value)
    if "price" in df.columns:
        df["price"] = df["price"].apply(normalize_price)
    else:
        df["price"] = np.nan

    # 3️⃣ Xác định cột gộp description
    exclude_cols = set(keep_cols)
    desc_cols = [c for c in df.columns if c not in exclude_cols]

    print("📌 Các cột sẽ gộp vào description:", desc_cols)

    # 4️⃣ Gộp description
    descriptions = []
    for _, row in df[desc_cols].iterrows():
        parts = []
        for col in desc_cols:
            val = row[col]
            if pd.notna(val) and str(val).strip() != "":
                parts.append(f"{col}: {val}")
        descriptions.append(" | ".join(parts))

    df[desc_col_name] = descriptions

    # 5️⃣ Giữ đúng schema & thứ tự
    final_cols = ["id", "name", "price", "image_url", "category", desc_col_name]
    df = df[final_cols]

    df.to_excel(output_path, index=False)
    print(f"✅ Đã lưu file: {output_path}")



In [3]:
import pandas as pd

def check_keep_columns(files, keep_cols):
    """
    Kiểm tra các cột trong từng file Excel:
    - In ra toàn bộ cột
    - Báo thiếu / dư so với keep_cols
    """

    print("🔍 KIỂM TRA CỘT TRƯỚC KHI PREPROCESS\n")

    for file in files:
        df = pd.read_excel(file)
        cols = list(df.columns)

        missing = [c for c in keep_cols if c not in cols]
        extra = [c for c in cols if c not in keep_cols]

        print(f"📂 File: {file}")
        print("   ➤ Các cột hiện có:", cols)

        if missing:
            print("   ❌ Thiếu keep_cols:", missing)
        else:
            print("   ✅ Đủ keep_cols")

        print("   ➕ Cột dư (sẽ gộp vào description):", extra)
        print("-" * 80)


In [4]:
files_to_check = [
    "chuotmaytinh.xlsx",
    "maygiat.xlsx",
    "sacduphong.xlsx",
    "maylanh.xlsx",
]

keep_cols = ["id", "name", "price", "image_url", "category"]

check_keep_columns(files_to_check, keep_cols)


🔍 KIỂM TRA CỘT TRƯỚC KHI PREPROCESS

📂 File: chuotmaytinh.xlsx
   ➤ Các cột hiện có: ['id', 'price', 'name', 'Loại chuột', 'Tương thích', 'Cảm biến', 'Độ phân giải tối đa:', 'Polling Rate:', 'Switch:', 'Feet:', 'Cách kết nối:', 'Độ dài dây/Khoảng cách kết nối:', 'Số nút bấm:', 'RGB:', 'Loại pin:', 'Cổng sạc:', 'category', 'Mô tả', 'image_url']
   ✅ Đủ keep_cols
   ➕ Cột dư (sẽ gộp vào description): ['Loại chuột', 'Tương thích', 'Cảm biến', 'Độ phân giải tối đa:', 'Polling Rate:', 'Switch:', 'Feet:', 'Cách kết nối:', 'Độ dài dây/Khoảng cách kết nối:', 'Số nút bấm:', 'RGB:', 'Loại pin:', 'Cổng sạc:', 'Mô tả']
--------------------------------------------------------------------------------
📂 File: maygiat.xlsx
   ➤ Các cột hiện có: ['id', 'price', 'Loại máy giặt:', 'Lồng giặt:', 'Khối lượng giặt:', 'Số người sử dụng:', 'Vệ sinh lồng giặt', 'Tốc độ quay vắt tối đa:', 'Chất liệu lồng giặt:', 'Chất liệu vỏ máy:', 'Chất liệu nắp máy:', 'Sản xuất tại:', 'Năm ra mắt:', 'Thời gian bảo hành động 

In [5]:
preprocess_product_file(
    "chuotmaytinh.xlsx",
    "chuotmaytinh_preprocessed.xlsx",
    keep_cols,
    "Chuột máy tính"
)

preprocess_product_file(
    "maygiat.xlsx",
    "maygiat_preprocessed.xlsx",
    keep_cols,
    "Máy giặt"
)

preprocess_product_file(
    "sacduphong.xlsx",
    "sacduphong_preprocessed.xlsx",
    keep_cols,
    "Sạc dự phòng"
)

preprocess_product_file(
    "maylanh.xlsx",
    "maylanh_preprocessed.xlsx",
    keep_cols,
    "Máy lạnh"
)

📂 Đang mở file: chuotmaytinh.xlsx
📌 Các cột sẽ gộp vào description: ['Loại chuột', 'Tương thích', 'Cảm biến', 'Độ phân giải tối đa:', 'Polling Rate:', 'Switch:', 'Feet:', 'Cách kết nối:', 'Độ dài dây/Khoảng cách kết nối:', 'Số nút bấm:', 'RGB:', 'Loại pin:', 'Cổng sạc:', 'Mô tả']
✅ Đã lưu file: chuotmaytinh_preprocessed.xlsx
📂 Đang mở file: maygiat.xlsx
📌 Các cột sẽ gộp vào description: ['Loại máy giặt:', 'Lồng giặt:', 'Khối lượng giặt:', 'Số người sử dụng:', 'Vệ sinh lồng giặt', 'Tốc độ quay vắt tối đa:', 'Chất liệu lồng giặt:', 'Chất liệu vỏ máy:', 'Chất liệu nắp máy:', 'Sản xuất tại:', 'Năm ra mắt:', 'Thời gian bảo hành động cơ:', 'Hiệu suất sử dụng điện:', 'Chương trình:', 'Công nghệ giặt', 'Bảng điều khiển:', 'Tiện ích', 'Kích thước - Khối lượng:', 'Chiều dài ống cấp nước:', 'Chiều dài ống thoát nước:', 'Ưu đãi lắp đặt', 'Tiện ích:', 'Hãng', 'Comment', 'Mô tả']
✅ Đã lưu file: maygiat_preprocessed.xlsx
📂 Đang mở file: sacduphong.xlsx
📌 Các cột sẽ gộp vào description: ['Dung lượng p

In [ ]:
keep_cols = ["id", "name", "price", "image_url", "category"]

preprocess_product_file(
    "tainghe.xlsx",
    "tainghe_preprocessed.xlsx",
    keep_cols,
    "Tai nghe"
)


In [ ]:
keep_cols_tablet = ["id", "name", "price", "image_url", "category"]

desc_cols_tablet = [
    'Thông tin pin', 'Tính năng', 'Tính năng đặc biệt', 'Điều khiển', 'Kết nối', 'Kích thước và khối lượng', 'Bảo hành', 'Công nghệ âm thanh', 'Tiện ích', 'Phím điều khiển', 'Mô tả'
]

preprocess_product_file(
    "tainghe.xlsx",
    "tainghe_preprocessed.xlsx",
    keep_cols_tablet,
    desc_cols_tablet
)

In [ ]:
keep_cols_tablet = ["id", "name", "price", "image_url", "category"]

desc_cols_tablet = [
    'Memory', 'Core clock', 'Giao diện bộ nhớ', 'Hãng sản xuất', 'Engine đồ họa', 'Chuẩn Bus', 'Clock bộ nhớ', 'Độ phân giải', 'Kết nối', 'Kích thước', 'Power Connectors', 'Mô tả'
]

preprocess_product_file(
    "vga.xlsx",
    "vga_preprocessed.xlsx",
    keep_cols_tablet,
    desc_cols_tablet
)

In [ ]:
keep_cols_tablet = ["id", "name", "price", "image_url", "category"]

desc_cols_tablet = [
    "Độ phân giải", "Thông số màn hình", "Hệ điều hành", "Vi xử lí",
    "Tốc độ cpu", "Card màn hình", "RAM", "Bộ nhớ",
    "Hỗ trợ thẻ nhớ", "Camera", "Quay phim", "Sạc nhanh"
]

preprocess_product_file(
    "tggd_tablet.xlsx",
    "tggd_tablet_preprocessed.xlsx",
    keep_cols_tablet,
    desc_cols_tablet
)


In [ ]:
keep_cols_pc = ["id", "name", "price", "image_url", "category"]
desc_cols_pc = ["CPU", "Mainboard", "RAM", "SSD", "PSU", "VGA", "Case", "Tản nhiệt"]
preprocess_product_file(
    "ttg_pc.xlsx",
    "ttg_pc_preprocessed.xlsx",
    keep_cols_pc,
    desc_cols_pc
)


In [ ]:

keep_cols = ["id", "name", "price", "image_url", "category"]

# Ví dụ với file điện thoại
desc_cols_phone = [
    "Hệ điều hành", "Chip", "Thông số chip", "Thông số", "Ram",
    "Bộ nhớ", "Camera", "Màn hình", "Độ phân giải", "Độ sáng màn hình",
    "Dung lượng pin", "Loại pin", "Sạc nhanh", "Chống nước",
    "Kết nối", "SIM", "Cổng sạc", "Chất liệu", "Hãng",
    "Quay video", "Tính năng đặc biệt", "Mô tả", "Comment"
]

preprocess_product_file(
    "phone_data.xlsx",
    "phone_data_preprocessed.xlsx",
    keep_cols,
    desc_cols_phone
)

In [6]:
def check_excel_column_order(files, expected_cols):
    """
    Kiểm tra thứ tự và tên cột của các file Excel
    """
    print("🔍 KIỂM TRA SCHEMA TRƯỚC KHI GỘP FILE\n")

    all_ok = True

    for file in files:
        df = pd.read_excel(file)
        cols = list(df.columns)

        print(f"📂 File: {file}")
        print("   ➤ Columns:", cols)

        if cols != expected_cols:
            all_ok = False
            print("   ❌ SAI schema hoặc thứ tự cột")
            print("   ➤ Mong đợi:", expected_cols)
        else:
            print("   ✅ ĐÚNG schema & thứ tự")

        print("-" * 80)

    return all_ok


In [8]:
expected_cols = [
    "id",
    "name",
    "price",
    "image_url",
    "category",
    "description"
]

files_to_merge = [
    "chuotmaytinh_preprocessed.xlsx",
    "maygiat_preprocessed.xlsx",
    "sacduphong_preprocessed.xlsx",
    "maylanh_preprocessed.xlsx",
]

if check_excel_column_order(files_to_merge, expected_cols):
    print("✅ TẤT CẢ FILE HỢP LỆ → CÓ THỂ GỘP")
else:
    print("❌ CÓ FILE SAI → KHÔNG NÊN GỘP")


🔍 KIỂM TRA SCHEMA TRƯỚC KHI GỘP FILE

📂 File: chuotmaytinh_preprocessed.xlsx
   ➤ Columns: ['id', 'name', 'price', 'image_url', 'category', 'description']
   ✅ ĐÚNG schema & thứ tự
--------------------------------------------------------------------------------
📂 File: maygiat_preprocessed.xlsx
   ➤ Columns: ['id', 'name', 'price', 'image_url', 'category', 'description']
   ✅ ĐÚNG schema & thứ tự
--------------------------------------------------------------------------------
📂 File: sacduphong_preprocessed.xlsx
   ➤ Columns: ['id', 'name', 'price', 'image_url', 'category', 'description']
   ✅ ĐÚNG schema & thứ tự
--------------------------------------------------------------------------------
📂 File: maylanh_preprocessed.xlsx
   ➤ Columns: ['id', 'name', 'price', 'image_url', 'category', 'description']
   ✅ ĐÚNG schema & thứ tự
--------------------------------------------------------------------------------
✅ TẤT CẢ FILE HỢP LỆ → CÓ THỂ GỘP


In [9]:
import pandas as pd

def merge_preprocessed_files(file_list, output_path):
    """
    file_list: danh sách đường dẫn các file excel đã preprocess
    output_path: file excel tổng sẽ xuất ra
    """

    dfs = []

    for file in file_list:
        try:
            df = pd.read_excel(file)

            # Kiểm tra các cột có đúng schema không
            required_cols = ["id", "name", "price", "image_url", "category", "description"]

            missing = [c for c in required_cols if c not in df.columns]
            if missing:
                print(f"⚠️ File {file} thiếu cột: {missing}. Bỏ qua file này.")
                continue

            dfs.append(df)
            print(f"✅ Đã nạp: {file}")

        except Exception as e:
            print(f"❌ Lỗi khi đọc {file}: {e}")

    # Nếu không có file hợp lệ
    if not dfs:
        print("❌ Không có file nào hợp lệ để gộp.")
        return

    # Gộp lại
    final_df = pd.concat(dfs, ignore_index=True)

    # Xuất file
    final_df.to_excel(output_path, index=False)

    print(f"✅ Đã gộp {len(dfs)} file lại thành 1.")
    print(f"📦 Xuất file: {output_path}")


In [11]:
merge_preprocessed_files(
    [
        "phone_data_preprocessed.xlsx",
        "laptop_data_preprocessed.xlsx",
        "tggd_tablet_preprocessed.xlsx",
        "ttg_pc_preprocessed.xlsx",
        "tainghe_preprocessed.xlsx",
        "vga_preprocessed.xlsx",
        "chuotmaytinh_preprocessed.xlsx",
        "maygiat_preprocessed.xlsx",
        "sacduphong_preprocessed.xlsx",
        "maylanh_preprocessed.xlsx"
        
    ],
    "raw_products.xlsx"
)

✅ Đã nạp: phone_data_preprocessed.xlsx
✅ Đã nạp: laptop_data_preprocessed.xlsx
✅ Đã nạp: tggd_tablet_preprocessed.xlsx
✅ Đã nạp: ttg_pc_preprocessed.xlsx
✅ Đã nạp: tainghe_preprocessed.xlsx
✅ Đã nạp: vga_preprocessed.xlsx
✅ Đã nạp: chuotmaytinh_preprocessed.xlsx
✅ Đã nạp: maygiat_preprocessed.xlsx
✅ Đã nạp: sacduphong_preprocessed.xlsx
✅ Đã nạp: maylanh_preprocessed.xlsx
✅ Đã gộp 10 file lại thành 1.
📦 Xuất file: raw_products.xlsx
